# Hamiltonian Variational Ansatz (HVA) — 2-Qubit Transverse-Field Ising Model

**Primitive:** Hamiltonian Variational Ansatz (HVA), 1 layer (p=1)

## SETUP (identical in both notebooks — Qiskit & CUDA-Q)

- **N_QUBITS** = 2
- **Target Hamiltonian:** H = -J·Z0Z1 - h·(X0 + X1), with **J = 1.0**, **h = 0.5**
- **Ansatz gate sequence** (fixed, non-optimized parameters for cross-verification):
  1. H(q0), H(q1) — prepare |++⟩, the ground state of the mixer term
  2. exp(-i·GAMMA·Z0Z1) — the "problem" layer, built from the ZZ interaction
  3. exp(-i·BETA·X0), exp(-i·BETA·X1) — the "mixer" layer
- **Fixed parameters:** GAMMA = 0.3, BETA = 0.4 (rad) — chosen once, never re-derived per framework
- **QUBIT ORDERING CONTRACT:** qubit 0 = **rightmost** bit in every printed bitstring
  (Qiskit's native convention). CUDA-Q natively prints qubit 0 as the **leftmost**
  bit — the CUDA-Q notebook explicitly reverses every bitstring to match this contract.
- **SHOTS** = 10000, **SEED** = 42
- **GLOBAL PHASE CONVENTION:** normalize so the first amplitude with `abs(amp) > 1e-9`,
  in ascending bitstring order (`00` → `01` → `10` → `11`), is rotated to be real and positive:
  `phase = angle(amp)`, then multiply the whole vector by `exp(-1j*phase)`.
- **Basis ordering for matrices:** {|00⟩, |01⟩, |10⟩, |11⟩} in ascending binary order.

## Theory recap

The Hamiltonian Variational Ansatz builds the circuit directly out of the terms of the
problem Hamiltonian, alternating a "problem" unitary with a "mixer" unitary — this is the
same structure as QAOA, but here it is not restricted to diagonal cost Hamiltonians.

For the 2-qubit transverse-field Ising model

  H = -J·Z₀Z₁ - h·(X₀ + X₁)

a single HVA layer is

  U(γ, β) = [ exp(-iβX₀)·exp(-iβX₁) ] · exp(-iγZ₀Z₁)

applied to the mixer ground state |++⟩ = H⊗H |00⟩.

- `exp(-iγZ₀Z₁)` is implemented as an `RZZ(2γ)` gate in Qiskit (`RZZ(θ) = exp(-iθ/2·Z⊗Z)`),
  and as the standard `CNOT · RZ(2γ) · CNOT` decomposition in CUDA-Q (verified below to be
  numerically identical to `exp(-iγZ⊗Z)`, not just equal up to global phase).
- `exp(-iβXᵢ)` is implemented as `RX(2β)` on each qubit (`RX(θ) = exp(-iθ/2·X)`).

**Ordering-convention warning:** CUDA-Q's `get_state()` and `sample()` index qubit 0 as the
most-significant (leftmost) bit — opposite to Qiskit. This is corrected explicitly in the
CUDA-Q notebook (see the dedicated "bit-order correction" cell) before any dictionary is
printed or compared.

**Accuracy check included:** since GAMMA/BETA are fixed (not variationally optimized), both
notebooks also compute ⟨ψ|H|ψ⟩ for this specific ansatz state and compare it against the
*exact* ground-state energy of H (via direct diagonalization with NumPy), to report what
fraction of the true ground energy this one fixed HVA layer captures.

**Run all cells top to bottom.**


In [ ]:
# Install Qiskit + Aer + the LaTeX-free circuit-drawing backend.
# Tested against: qiskit >= 1.0, qiskit-aer >= 0.14 (current stable release line).
# pylatexenc is required by circuit.draw(output="mpl") for rendering gate labels.
!pip install -q qiskit qiskit-aer pylatexenc


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# ---- Global constants (must match the CUDA-Q notebook exactly) ----
N_QUBITS = 2
J = 1.0            # ZZ coupling strength
H_FIELD = 0.5      # transverse field strength
GAMMA = 0.3        # problem-layer angle
BETA = 0.4         # mixer-layer angle
SHOTS = 10000
SEED = 42
TOPIC_SLUG = "hva_tfim"

print(f"N_QUBITS = {N_QUBITS}")
print(f"J = {J}, H_FIELD = {H_FIELD}")
print(f"GAMMA = {GAMMA}, BETA = {BETA}")
print(f"SHOTS = {SHOTS}, SEED = {SEED}")


In [ ]:
# Build the 1-layer Hamiltonian Variational Ansatz, gate by gate.
qc = QuantumCircuit(N_QUBITS, name="HVA_TFIM")

# --- Step 1: prepare the mixer ground state |++> ---
qc.h(0)   # qubit 0 -> |+>, the +1 eigenstate of X0
qc.h(1)   # qubit 1 -> |+>, the +1 eigenstate of X1

# --- Step 2: "problem" layer, exp(-i*GAMMA*Z0*Z1) ---
# Qiskit's RZZ(theta) implements exp(-i*theta/2 * Z⊗Z), so theta = 2*GAMMA
# gives exactly exp(-i*GAMMA*Z0*Z1).
qc.rzz(2 * GAMMA, 0, 1)

# --- Step 3: "mixer" layer, exp(-i*BETA*X0) and exp(-i*BETA*X1) ---
# Qiskit's RX(theta) implements exp(-i*theta/2 * X), so theta = 2*BETA
# gives exactly exp(-i*BETA*X) on each qubit.
qc.rx(2 * BETA, 0)
qc.rx(2 * BETA, 1)

print(qc)


In [ ]:
# Draw the circuit with matplotlib and save it to disk.
fig = qc.draw(output="mpl", style={"name": "default"})
fig.savefig(f"{TOPIC_SLUG}_qiskit_circuit.png", dpi=150, bbox_inches="tight")
fig


In [ ]:
# ---- (a) Build the full 4x4 unitary by hand in NumPy, gate-by-gate ----
# Basis ordering for these hand-built matrices: {|00>, |01>, |10>, |11>} ascending
# binary, with qubit 0 as the LAST (rightmost) tensor factor -- i.e. kron(op_q1, op_q0) --
# which is exactly Qiskit's own little-endian convention, so no reversal is needed here.

I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H2 = (1 / np.sqrt(2)) * np.array([[1, 1], [1, -1]], dtype=complex)

def RX(theta):
    return np.array([[np.cos(theta / 2), -1j * np.sin(theta / 2)],
                      [-1j * np.sin(theta / 2), np.cos(theta / 2)]], dtype=complex)

def RZZ(theta):
    ZZ = np.kron(Z, Z)
    diag = np.diag(ZZ).real
    return np.diag(np.exp(-1j * theta / 2 * diag))

print("H gate:\n", np.round(H2, 4))
print("RX(2*BETA):\n", np.round(RX(2 * BETA), 4))
print("RZZ(2*GAMMA) (diag, basis |00>,|01>,|10>,|11>):\n", np.round(RZZ(2 * GAMMA), 4))

U_H = np.kron(H2, H2)
U_ZZ = RZZ(2 * GAMMA)
U_RX = np.kron(RX(2 * BETA), RX(2 * BETA))
U_hand = U_RX @ U_ZZ @ U_H

print("\nFull hand-built operator U_hand = RX_layer . RZZ_layer . H_layer:\n", np.round(U_hand, 4))

# ---- (b) Extract the same operator directly from the Qiskit circuit ----
U_qiskit = Operator(qc).data

print("\nQiskit-extracted Operator(qc):\n", np.round(U_qiskit, 4))

match = np.allclose(U_hand, U_qiskit, atol=1e-8)
print(f"\nHand-built vs Qiskit-extracted operator match (atol=1e-8): {match}")
assert match, "Hand-built and Qiskit-extracted operators disagree!"


In [ ]:
def normalize_global_phase(statevector):
    """Rotate so the first amplitude with abs(amp) > 1e-9, in ascending bitstring
    order, is real and positive. Shared convention -- identical in both notebooks."""
    amps = np.array(statevector, dtype=complex)
    for amp in amps:
        if abs(amp) > 1e-9:
            phase = np.angle(amp)
            return amps * np.exp(-1j * phase)
    return amps

# Exact statevector from the (unmeasured) circuit.
sv = Statevector.from_instruction(qc)
raw_amps = sv.data  # Qiskit already orders qubit 0 as the rightmost (LSB) bit -- no reversal needed

norm_amps = normalize_global_phase(raw_amps)

basis_labels = [format(i, f"0{N_QUBITS}b") for i in range(2 ** N_QUBITS)]  # ascending binary
probs = np.abs(norm_amps) ** 2

print("Phase-normalized amplitudes (qubit 0 = rightmost bit):")
for label, amp in zip(basis_labels, norm_amps):
    print(f"  |{label}>: {amp:.6f}")

QISKIT_STATEVECTOR_PROBS = {label: float(p) for label, p in zip(basis_labels, probs)}
print("\nExact probability dictionary:")
for k, v in QISKIT_STATEVECTOR_PROBS.items():
    print(f"  {k}: {v:.6f}")
print(f"\nSum of probabilities: {sum(QISKIT_STATEVECTOR_PROBS.values()):.8f}")


### Ordering-fix note (Qiskit)

**No bitstring reversal was needed in this notebook.** Qiskit already indexes
`Statevector.data` and measurement-count keys with qubit 0 as the least-significant
(rightmost) bit, which is exactly the shared QUBIT ORDERING CONTRACT for this pair of
notebooks. The only transformation applied was the shared **global phase normalization**
(rotate so the first nonzero amplitude in ascending-bitstring order is real and positive).


In [ ]:
# Add measurements and sample with the shared SEED / SHOTS.
qc_meas = qc.copy()
qc_meas.measure_all()

sim = AerSimulator(seed_simulator=SEED)
qc_transpiled = transpile(qc_meas, sim)
job = sim.run(qc_transpiled, shots=SHOTS, seed_simulator=SEED)
result = job.result()
counts = result.get_counts()

# Qiskit's count keys already use qubit 0 = rightmost bit -- matches the shared
# convention, so no reordering is needed here (see the ordering-fix note above).
print("Raw counts:", counts)

QISKIT_MEASURED_PROBS = {label: counts.get(label, 0) / SHOTS for label in basis_labels}
print("\nMeasured probability dictionary:")
for k, v in QISKIT_MEASURED_PROBS.items():
    print(f"  {k}: {v:.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(QISKIT_MEASURED_PROBS.keys(), QISKIT_MEASURED_PROBS.values(),
       color="#4285f4", edgecolor="black")
ax.set_ylim(0, 1)
ax.set_xlabel("Bitstring (qubit 0 = rightmost bit)")
ax.set_ylabel("Probability")
ax.set_title(f"HVA_TFIM (Qiskit) -- measured, {SHOTS} shots, seed={SEED}")
for label, p in QISKIT_MEASURED_PROBS.items():
    ax.text(label, p + 0.01, f"{p:.3f}", ha="center", fontsize=9)
fig.tight_layout()
fig.savefig(f"{TOPIC_SLUG}_qiskit_histogram.png", dpi=150, bbox_inches="tight")
fig


In [ ]:
print(f"Circuit depth: {qc.depth()}")
print(f"Gate count (by type): {dict(qc.count_ops())}")
print(f"Total operations (size): {qc.size()}")


In [ ]:
# ---- Accuracy check: how close is this fixed (non-optimized) HVA layer to the ----
# ---- true ground state of H = -J*Z0Z1 - h*(X0+X1) ? ----
ZZ = np.kron(Z, Z)
X0 = np.kron(I2, X)  # qubit 0 operator (rightmost tensor factor, matches our convention)
X1 = np.kron(X, I2)  # qubit 1 operator
Ham = -J * ZZ - H_FIELD * (X0 + X1)

psi = sv.data  # un-normalized-phase exact statevector is fine for an expectation value
energy = np.real(np.conj(psi) @ Ham @ psi)

exact_eigs = np.linalg.eigvalsh(Ham)
exact_ground_energy = exact_eigs.min()

print(f"<H> for this fixed HVA(GAMMA={GAMMA}, BETA={BETA}) state: {energy:.6f}")
print(f"Exact ground-state energy (NumPy eigh):              {exact_ground_energy:.6f}")
print(f"Fraction of ground energy captured: {energy / exact_ground_energy:.4%}")


In [ ]:
print("=== Convention notes (Qiskit) ===")
print("Bit order  : qubit 0 = rightmost bit -- Qiskit native, no reversal applied.")
print("Phase      : first nonzero amplitude in ascending bitstring order rotated to be real & positive.")
print("Shots/seed : SHOTS =", SHOTS, " SEED =", SEED, " (AerSimulator(seed_simulator=SEED))")


In [ ]:
# Final result dictionary -- built from the EXACT statevector probabilities
# (deterministic, so this is what the cross-notebook diff-check compares).
QISKIT_RESULT = QISKIT_STATEVECTOR_PROBS
print("QISKIT_RESULT =", QISKIT_RESULT)


In [ ]:
# ---- Cross-notebook diff-check (commented out: CUDAQ_RESULT only exists if you ----
# ---- run the CUDA-Q notebook in the same session, or paste its QISKIT_RESULT dict here) ----
#
# max_delta = max(abs(QISKIT_RESULT[k] - CUDAQ_RESULT[k]) for k in QISKIT_RESULT)
# assert max_delta < 1e-3, f"Mismatch between Qiskit and CUDA-Q results: {max_delta}"
# print(f"Max probability delta between QISKIT_RESULT and CUDAQ_RESULT: {max_delta:.2e}")

# Hand-computed expected values for sanity-checking (derived independently with NumPy,
# see the "matrix representation" and "exact statevector" cells above for the full derivation):
EXPECTED_PROBS = {"00": 0.391100, "01": 0.108900, "10": 0.108900, "11": 0.391100}
EXPECTED_ENERGY = -1.389737   # <H> for GAMMA=0.3, BETA=0.4 (fixed, non-optimized ansatz)
EXPECTED_GROUND_ENERGY = -1.414214  # exact ground energy of H = -J*Z0Z1 - h*(X0+X1), J=1, h=0.5

max_delta_vs_hand = max(abs(QISKIT_RESULT[k] - EXPECTED_PROBS[k]) for k in QISKIT_RESULT)
print(f"Max probability delta vs. hand-computed expected values: {max_delta_vs_hand:.2e}")
assert max_delta_vs_hand < 1e-3, "Qiskit result disagrees with the hand-computed expectation!"
print("PASSED: Qiskit result matches the hand-computed expected probabilities within 1e-3.")
